### `create_react_agent`를 사용하는 이유

#### 1. 정의
- **`create_react_agent`**는  
  LangChain에서 제공하는 **React(Reason + Act) 에이전트**를  
  손쉽게 생성할 수 있는 함수입니다.
- LLM + 여러 도구(tool) + 프롬프트를 결합해  
  **자동으로 멀티스텝 추론과 도구 실행이 반복되는 에이전트 구조**를 만들어줍니다.

#### 2. 사용하는 주요 이유
##### 2.1 LLM의 능동적 문제 해결 프레임워크 제공
- LLM이 단순 응답을 넘어서  
  **스스로 "생각→도구 실행→관찰→최종 답"의 루프를 반복**하도록 설계

##### 2.2 도구와 LLM을 유연하게 연결
- **여러 개의 외부 도구(계산기, 검색, API 등)**를  
  LLM이 필요에 따라 직접 선택·입력값 생성·호출·결과 해석을 자동으로 수행

##### 2.3 표준화된 멀티스텝 추론 패턴 지원
- "Thought-Action-Observation" 패턴을  
  프롬프트에 자동 삽입·누적  
  → 멀티스텝 reasoning/workflow가 쉬워짐

##### 2.4 프롬프트·도구 조합 자동화
- 별도 복잡한 체인 조립 없이  
  LLM, 도구, 프롬프트만 넘기면  
  **최적화된 에이전트가 즉시 완성**

##### 2.5 디버깅/투명성/확장성
- reasoning 과정, 도구 사용 과정이 verbose 모드에서 모두 출력  
- 실시간 로그 추적, 디버깅, 분석이 용이  
- 도구 추가·변경·조합이 유연

#### 3. 실무적 장점
- **AI Copilot, 자동화 챗봇, 분석·요약·검색 등  
  복잡한 워크플로우를 LLM+도구 조합만으로 빠르게 구현** 가능
- 도구가 늘어나도 일관된 인터페이스 유지
- 최신 OpenAI Function Call/Tool Calling과도 구조적으로 유사

#### 4. 한 줄 요약
> **`create_react_agent`는  
> LLM이 멀티스텝 추론을 하며  
> 외부 도구를 능동적으로 사용해 문제를 자동 해결할 수 있도록  
> “최적화된 에이전트 아키텍처”를 코드 한 줄로 빠르게 구현하는 LangChain의 핵심 함수입니다.**

In [ ]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain.agents import create_react_agent, AgentExecutor, tool

# 환경변수 로드
load_dotenv()

# Gemini LLM 초기화
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0
)

#### 2. 동작 원리 및 구조

- **프롬프트 템플릿 내에**
    - Question: 사용자 입력
    - Thought: 다음 행동(혹은 의사결정) 전, 스스로 생각을 표현
    - Action: 사용할 도구 지정(예: 검색, 계산, 외부 API 등)
    - Action Input: 도구에 실제로 입력할 데이터
    - Observation: 도구 실행 결과를 받아 적음
    - (반복)
    - Thought: (필요하면 추가 행동 반복)
    - Final Answer: 최종 답변

- **LLM이 이 구조대로 단계별로 응답을 생성하며,  
  매 Thought/Action/Observation 루프마다 실제 도구가 자동 실행되고  
  그 결과가 다시 LLM의 다음 Thought에 반영됨**
#### 3. 실전 예시

- Question: 2024년 크리스마스는 무슨 요일인가요?
- Thought: 날짜 계산이 필요하다. calculator 도구를 사용해야겠다.
- Action: calculator
- Action Input: 2024년 12월 25일은 무슨 요일?
- Observation: 2024년 12월 25일은 수요일입니다.
- Thought: 이제 최종 답을 알겠습니다.
- Final Answer: 2024년 크리스마스는 수요일입니다.

In [ ]:
# 도구 정의 (LangChain의 @tool 데코레이터 사용)
@tool("calculator", return_direct=True)
def calculator(query: str) -> str:
    """수학 계산을 수행합니다. 입력: 수학 표현식 (예: '2+2', '10*5')"""
    try:
        result = eval(query)
        return f"결과: {result}"
    except:
        return "계산 오류가 발생했습니다."

tools = [calculator]

# React 프롬프트 템플릿
prompt = PromptTemplate.from_template("""
다음 도구들을 사용하여 질문에 답해주세요:

{tools}

사용 가능한 도구 이름: {tool_names}
다음 형식을 사용하세요:

Question: 입력 질문
Thought: 무엇을 해야 하는지 생각
Action: 사용할 도구
Action Input: 도구에 전달할 입력
Observation: 도구의 결과
... (필요시 Thought/Action/Action Input/Observation 반복)
Thought: 이제 최종 답을 알겠습니다
Final Answer: 최종 답변

Question: {input}
Thought: {agent_scratchpad}
""")

# React Agent 생성
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

# 에이전트 실행기(Executor) 생성
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# 에이전트에게 실제 질문해보기
result = agent_executor.invoke({"input": "23 * 7은 얼마인가요?"})
print("\n=== 에이전트 답변 ===")
print(result["output"])

result2 = agent_executor.invoke({"input": "100 + 2345는?"})
print("\n=== 두 번째 에이전트 답변 ===")
print(result2["output"])